# Lab 2.1 — Comparing LLMs Across Providers

## Introduction

This is the **first lab** in the AI Models module. In the Foundations labs, you learned to call a single LLM through OpenAI. Here, you expand that skill by working with **multiple model providers** and comparing how they respond to the same prompt.

The workflow has two phases:
1. Use one model to generate a challenging evaluation question
2. Send that same question to **OpenAI**, **Anthropic (Claude)**, and **Google (Gemini)** and compare the answers side by side

You will also see that different providers expose slightly different APIs — but Gemini can be accessed through an **OpenAI-compatible endpoint**, letting you reuse the same client pattern you already know.

Before starting, make sure you have:
- Completed the Foundations labs and Setup (`setup/2_api_key_setup.ipynb`)
- Installed dependencies from `2_ai_models/requirements.txt`
- At minimum, an OpenAI API key in your `.env` file (Anthropic and Google keys are optional but needed for their respective sections)

---

## Intention (Learning Objectives)

By the end of this lab, you should be able to:

1. **Load multiple API keys** — use `python-dotenv` and verify keys for OpenAI, Anthropic, and Google
2. **Generate a shared benchmark prompt** — use one LLM to create a question, then reuse it across models
3. **Call OpenAI models** — send the same `messages` payload to `gpt-4o-mini` and display the response
4. **Call Anthropic models** — use the Anthropic SDK and understand differences such as required `max_tokens`
5. **Call Gemini via an OpenAI-compatible API** — point the OpenAI client at Google's base URL with your Google API key
6. **Collect and compare results** — store model names and answers in parallel lists for side-by-side evaluation

---

## Lab Structure

| Part | Content |
|------|---------|
| **Part 1** | Imports, load environment variables, verify API keys |
| **Part 2** | Generate a challenging evaluation question with GPT-4o-mini |
| **Part 3** | Prepare shared state — reset messages and initialize result lists |
| **Part 4** | Ask the same question to OpenAI, Anthropic, and Gemini |

> **Note:** Run cells top to bottom in order. Anthropic and Google sections can be skipped if you do not have those API keys configured.

## Part 1 — Setup & Verify API Keys

Import the packages we need, load variables from `.env`, and confirm which provider keys are available.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [ ]:
# Always remember to do this!
load_dotenv(override=True)

In [ ]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

## Part 2 — Generate an Evaluation Question

Ask GPT-4o-mini to create a challenging question. We will send this same question to every model in Part 4 so we can compare their answers fairly.

In [ ]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [ ]:
messages

In [ ]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)

## Part 3 — Prepare for Model Comparison

Reset `messages` with the generated question and create empty lists to collect each model's name and answer.

In [ ]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

## Part 4 — Ask the Same Question to Multiple LLMs

Run the cells below to query each provider. Notice how the APIs differ — especially Anthropic's required `max_tokens` parameter and Gemini's OpenAI-compatible base URL.

### OpenAI — `gpt-4o-mini`

In [ ]:
# The API we know well

model_name = "gpt-4o-mini"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

### Anthropic — `claude-3-7-sonnet-latest`

In [ ]:
# Anthropic has a slightly different API, and max_tokens is required

model_name = "claude-3-7-sonnet-latest"

claude = Anthropic()
response = claude.messages.create(model=model_name, messages=messages, max_tokens=1000)
answer = response.content[0].text

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

### Google Gemini — `gemini-2.0-flash` (via OpenAI-compatible API)

In [ ]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.0-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

---

## Next Steps

Review the answers stored in `competitors` and `answers`. Which model gave the most thoughtful response? Try swapping in different model names, adding a fourth provider, or asking the LLM to rank the answers it received.